In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as nca
import xarray as xr
import pandas as pd
import os

In [ ]:
base_dir = '/scratch/gpfs/GEOCLIM/synda/data/CMIP6/CMIP/'
save_dir = '/scratch/gpfs/GEOCLIM/LRGROUP/db9274/Analysis/Variability_quantification/NETCDF/FFT_nodetrending_subsampled_Nquist_handled/'
subsampled_dir = '/scratch/gpfs/GEOCLIM/LRGROUP/db9274/Analysis/Variability_quantification/Ito_2022_OI_algorithm/optint_wod_o2/'

subdir_oxygen = ['Omon/o2/gr/', 'Omon/o2/gr/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gn/', 'Omon/o2/gr/', 'Omon/o2/gr/']
subdir_temp = ['Omon/thetao/gr/', 'Omon/thetao/gr/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gn/', 'Omon/thetao/gr/', 'Omon/thetao/gr/']
subdir_salt = ['Omon/so/gr/', 'Omon/so/gr/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gn/', 'Omon/so/gr/', 'Omon/so/gr/']
expt = ['NOAA-GFDL/GFDL-CM4/', 'NOAA-GFDL/GFDL-ESM4/', 'CCCma/CanESM5/', 'CCCma/CanESM5-CanOE/', 'CSIRO/ACCESS-ESM1-5/', 'MIROC/MIROC-ES2L/', 'MOHC/UKESM1-0-LL/', 'IPSL/IPSL-CM6A-LR/', 'MRI/MRI-ESM2-0/', 'MPI-M/MPI-ESM1-2-LR/', 'MPI-M/MPI-ESM1-2-HR/', 'CNRM-CERFACS/CNRM-ESM2-1/', 'NCC/NorESM2-LM/', 'NCC/NorESM2-MM/']
expt_name = ['GFDL_CM4', 'GFDL_ESM4', 'CanESM5', 'CanESM5_CanOE', 'ACCESS_ESM1.5', 'MIROC_ES2L', 'UKESM1.0LL', 'IPSL_CM6A_LR', 'MRI_ESM2_0', 'MPI_ESM1_2_LR', 'MPI_ESM1_2_HR', 'CNRM_ESM2_1', 'NorESM2_LM', 'NorESM2_MM']
scenario_name = 'historical'
ensembles = [['r1i1p1f1'], 
             ['r1i1p1f1'], 
             ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'], 
             ['r1i1p2f1', 'r2i1p2f1', 'r3i1p2f1'], 
             ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'], 
             ['r1i1p1f2', 'r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r5i1p1f2'], 
             ['r1i1p1f2', 'r2i1p1f2', 'r3i1p1f2'], 
             ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'], 
             ['r1i2p1f1'], 
             ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'], 
             ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'], 
             ['r1i1p1f2', 'r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r5i1p1f2'], 
             ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'], 
             ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1']]
version = [['v20180701'], 
           ['v20190726'], 
           ['v20190429', 'v20190429', 'v20190429', 'v20190429', 'v20190429'], 
           ['v20190429', 'v20190429', 'v20190429'], 
           ['v20191115', 'v20191128', 'v20191203'], 
           ['v20190823', 'v20190823', 'v20190823', 'v20200318', 'v20200318'],
           ['v20190627', 'v20190708', 'v20190708'], 
           ['v20180803', 'v20190305', 'v20180803', 'v20180803', 'v20180803'], 
           ['v20210311'], 
           ['v20190710', 'v20190710', 'v20190710', 'v20190710', 'v20190710'], 
           ['v20190710', 'v20190710', 'v20190710', 'v20190710', 'v20190710'], 
           ['v20181206', 'v20190125', 'v20190125', 'v20190125', 'v20190125'], 
           ['v20190815', 'v20190920', 'v20190920'],
           ['v20191108', 'v20200218', 'v20200702']]

version_ssp245 = [['v20180701'], 
                  ['v20180701'], 
                  ['v20190429', 'v20190429', 'v20190429', 'v20190429', 'v20190429'], 
                  ['v20190429', 'v20190429', 'v20190429'], 
                  ['v20191115', 'v20191128', 'v20191203'], 
                  ['v20190823', 'v20190823', 'v20190823', 'v20200318', 'v20200318'],
                  ['v20190507', 'v20190718', 'v20190801'], 
                  ['v20190119', 'v20190305', 'v20180803', 'v20180803', 'v20180803'], 
                  ['v20210311'], 
                  ['v20190710', 'v20190710', 'v20190710', 'v20190710'], 
                  ['v20190710', 'v20190710', 'v20190710', 'v20190710', 'v20190710'], 
                  ['v20190328', 'v20190125', 'v20190125', 'v20190125', 'v20190125'], 
                  ['v20191108', 'v20190920', 'v20190920'],
                  ['v20191108', 'v20200218', 'v20200702']]

start_time, end_time = '1965-01-01', '2021-12-30'
basin_names = ['Atlantic', 'Pacific', 'Indian', 'Southern', 'Arctic']
basin_nums = [1, 2, 3, 10, 11]

In [ ]:
def fix_lon_0_360(da, lon_name="x"):
    lon = da[lon_name]
    lon_fixed = (lon % 360)
    return da.assign_coords({lon_name: lon_fixed}).sortby(lon_name)

In [ ]:
projection_dir = '/scratch/gpfs/GEOCLIM/LRGROUP/db9274/Analysis/Variability_quantification/Ito_2024_ML_algorithm/ML4O2_Dhruv_potden/FINAL/'

## 1a. Merge all basins to get subsampled oxygen product

In [ ]:
j = 0
algorithm = 'NN'

In [ ]:
for i in range(14):
    Atlantic_oxygen = xr.open_dataset(projection_dir + expt_name[i] + f'/projected_o2_{algorithm}_' + basin_names[0] + '_' + expt_name[i] + '_' + ensembles[i][j] + '.nc')['o2']
    Pacific_oxygen = xr.open_dataset(projection_dir + expt_name[i] + f'/projected_o2_{algorithm}_' + basin_names[1] + '_' + expt_name[i] + '_' + ensembles[i][j] + '.nc')['o2']
    Indian_oxygen = xr.open_dataset(projection_dir + expt_name[i] + f'/projected_o2_{algorithm}_' + basin_names[2] + '_' + expt_name[i] + '_' + ensembles[i][j] + '.nc')['o2']
    Southern_oxygen = xr.open_dataset(projection_dir + expt_name[i] + f'/projected_o2_{algorithm}_' + basin_names[3] + '_' + expt_name[i] + '_' + ensembles[i][j] + '.nc')['o2']
    Arctic_oxygen = xr.open_dataset(projection_dir + expt_name[i] + f'/projected_o2_{algorithm}_' + basin_names[4] + '_' + expt_name[i] + '_' + ensembles[i][j] + '.nc')['o2']
    oxygen_map = Atlantic_oxygen.fillna(0) + Pacific_oxygen.fillna(0) + Indian_oxygen.fillna(0) + Southern_oxygen.fillna(0) + Arctic_oxygen.fillna(0)
    oxygen_map = oxygen_map.where(oxygen_map!=0)
    
    basin_mask = xr.open_dataset('/scratch/gpfs/GEOCLIM/LRGROUP/db9274/Analysis/Variability_quantification/Ito_2022_OI_algorithm/optint_wod_o2/basin_mask_01.nc')["basin_mask"].isel(depth = 0)
    basin_mask = basin_mask.rename({'lon': 'x', 'lat': 'y'})
    basin_mask = fix_lon_0_360(basin_mask)
    basin_mask = xr.where(basin_mask.isin(basin_nums), 1, np.nan)
    
    oxygen_map = oxygen_map * basin_mask
    oxygen_map.rename('o2').to_netcdf(projection_dir + f'../NETCDF/OSD_CTD_Argo_{algorithm}/' + expt_name[i] + '_' + ensembles[i][0] + '_oxygen_gridded.nc')
    print('Merging completed for ' + expt_name[i] + '.')

## 1b. Merge basins for observations

In [ ]:
algorithm = 'NN'
Atlantic_oxygen = xr.open_dataset(projection_dir + f'Observations/projected_o2_{algorithm}_' + basin_names[0] + '_Observations.nc')['o2']
Pacific_oxygen = xr.open_dataset(projection_dir + f'Observations/projected_o2_{algorithm}_' + basin_names[1] + '_Observations.nc')['o2']
Indian_oxygen = xr.open_dataset(projection_dir + f'Observations/projected_o2_{algorithm}_' + basin_names[2] + '_Observations.nc')['o2']
Southern_oxygen = xr.open_dataset(projection_dir + f'Observations/projected_o2_{algorithm}_' + basin_names[3] + '_Observations.nc')['o2']
Arctic_oxygen = xr.open_dataset(projection_dir + f'Observations/projected_o2_{algorithm}_' + basin_names[4] + '_Observations.nc')['o2']
oxygen_map = Atlantic_oxygen.fillna(0) + Pacific_oxygen.fillna(0) + Indian_oxygen.fillna(0) + Southern_oxygen.fillna(0) + Arctic_oxygen.fillna(0)
#oxygen_map = oxygen_map.where(oxygen_map > 0)

basin_mask = xr.open_dataset('/scratch/gpfs/GEOCLIM/LRGROUP/db9274/Analysis/Variability_quantification/Ito_2022_OI_algorithm/optint_wod_o2/basin_mask_01.nc')["basin_mask"].isel(depth = 0)
basin_mask = basin_mask.rename({'lon': 'x', 'lat': 'y'})
basin_mask = fix_lon_0_360(basin_mask)
basin_mask = xr.where(basin_mask.isin(basin_nums), 1, np.nan)

oxygen_map = oxygen_map * basin_mask
oxygen_map = xr.where(oxygen_map!=0, oxygen_map, np.nan)
oxygen_map.rename('o2').to_netcdf(projection_dir + f'../NETCDF/OSD_CTD_Argo_{algorithm}/Observations_oxygen_gridded_1965_2018.nc')

In [ ]:
algorithm = 'RF'
Atlantic_oxygen = xr.open_dataset(projection_dir + f'Observations_EN4/projected_o2_{algorithm}_' + basin_names[0] + '_Observations_EN4.nc')['o2']
Pacific_oxygen = xr.open_dataset(projection_dir + f'Observations_EN4/projected_o2_{algorithm}_' + basin_names[1] + '_Observations_EN4.nc')['o2']
Indian_oxygen = xr.open_dataset(projection_dir + f'Observations_EN4/projected_o2_{algorithm}_' + basin_names[2] + '_Observations_EN4.nc')['o2']
Southern_oxygen = xr.open_dataset(projection_dir + f'Observations_EN4/projected_o2_{algorithm}_' + basin_names[3] + '_Observations_EN4.nc')['o2']
Arctic_oxygen = xr.open_dataset(projection_dir + f'Observations_EN4/projected_o2_{algorithm}_' + basin_names[4] + '_Observations_EN4.nc')['o2']
oxygen_map = Atlantic_oxygen.fillna(0) + Pacific_oxygen.fillna(0) + Indian_oxygen.fillna(0) + Southern_oxygen.fillna(0) + Arctic_oxygen.fillna(0)
#oxygen_map = oxygen_map.where(oxygen_map > 0)

basin_mask = xr.open_dataset('/scratch/gpfs/GEOCLIM/LRGROUP/db9274/Analysis/Variability_quantification/Ito_2022_OI_algorithm/optint_wod_o2/basin_mask_01.nc')["basin_mask"].isel(depth = 0)
basin_mask = basin_mask.rename({'lon': 'x', 'lat': 'y'})
basin_mask = fix_lon_0_360(basin_mask)
basin_mask = xr.where(basin_mask.isin(basin_nums), 1, np.nan)

oxygen_map = oxygen_map * basin_mask
oxygen_map = xr.where(oxygen_map!=0, oxygen_map, np.nan)
oxygen_map.rename('o2').to_netcdf(projection_dir + f'../NETCDF/OSD_CTD_Argo_{algorithm}/Observations_EN4_oxygen_gridded.nc')

#### 2. Machine learning parameters and predictions for each basin

In [ ]:
RF_parameters = {"n_estimators": [50, 100, 200, 400, 600], "min_samples_split": [2, 5, 10]}

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(nrows = 2, ncols = 3, figsize = (16, 7))

for k in range(5):
    evaluation_summary = np.load(projection_dir + expt_name[0] + '/evaluation_summary_RF_' + basin_names[k] + '_GFDL_CM4_r1i1p1f1.npz')
    mean_R2 = evaluation_summary['mean_RMSE']

    sns.heatmap(mean_R2, annot=False, cmap="YlGnBu", fmt=".1f", ax = axes[k//3][k%3])
    axes[k//3][k%3].set_xticklabels(RF_parameters['n_estimators'])
    axes[k//3][k%3].set_yticklabels(RF_parameters['min_samples_split'])
    axes[k//3][k%3].set_xlabel('Number of estimators')
    axes[k//3][k%3].set_ylabel('Minimum sample split')
    axes[k//3][k%3].set_title(basin_names[k] + ' Ocean')

fig.tight_layout()

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(nrows = 2, ncols = 3, figsize = (16, 7))

for k in range(5):
    oxygen_fit = np.load(projection_dir + expt_name[0] + '/o2fit_pred_RF_' + basin_names[k] + '_GFDL_CM4_r1i1p1f1.npz')
    truth = oxygen_fit['truth']
    est = oxygen_fit['est']

    axes[k//3][k%3].scatter(truth, est)
    axes[k//3][k%3].set_xlabel("Earth system Model's simulated oxygen")
    axes[k//3][k%3].set_ylabel("Random forest's simulated oxygen")
    axes[k//3][k%3].set_title(basin_names[k] + ' Ocean')

fig.tight_layout()

## 3. Machine learning prediction for observations

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(nrows = 2, ncols = 3, figsize = (16, 7))

for k in range(5):
    evaluation_summary = np.load(projection_dir + 'Observations' + '/evaluation_summary_RF_' + basin_names[k] + '_Observations.npz')
    mean_R2 = evaluation_summary['mean_RMSE']

    sns.heatmap(mean_R2, annot=False, cmap="YlGnBu", fmt=".1f", ax = axes[k//3][k%3])
    axes[k//3][k%3].set_xticklabels(RF_parameters['n_estimators'])
    axes[k//3][k%3].set_yticklabels(RF_parameters['min_samples_split'])
    axes[k//3][k%3].set_xlabel('Number of estimators')
    axes[k//3][k%3].set_ylabel('Minimum sample split')
    axes[k//3][k%3].set_title(basin_names[k] + ' Ocean')

fig.tight_layout()

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(nrows = 2, ncols = 3, figsize = (16, 7))

for k in range(5):
    oxygen_fit = np.load(projection_dir + 'Observations' + '/o2fit_pred_RF_' + basin_names[k] + '_Observations.npz')
    truth = oxygen_fit['truth']
    est = oxygen_fit['est']

    axes[k//3][k%3].scatter(truth, est)
    axes[k//3][k%3].set_xlabel("Earth system Model's simulated oxygen")
    axes[k//3][k%3].set_ylabel("Random forest's simulated oxygen")
    axes[k//3][k%3].set_title(basin_names[k] + ' Ocean')

fig.tight_layout()